In [1]:
import gc, json, os, sys
import numpy as np
import torch
import ot
from scipy.spatial.distance import cdist

sys.path.insert(0, os.path.abspath(".."))
from models import Domain, DomainFeatures
from DatasetFeatureLoader import DatasetFeatureLoader
from FeatureCache import FeatureCache
from universal_feature_extractor import UniversalFeatureExtractor
from utils import make_deterministic

/opt/conda/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/conda/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [ ]:
SEED = 42
N_SAMPLES = 1000  # features loaded per domain
N_CALIBRATION = 300  # samples per domain used in each Sinkhorn test
N_EXTREME = 5  # cheapest + most expensive domain pairs to test
OUTPUT_DIR = (
    "/workspace/wasserstein-metric/pretraining_pipeline_results_euclidean_only/"
)

DOMAINS = [
    Domain("SAR-QXSLAB", "/workspace/mnt-data/QXSLAB_SAROPT/sar_256_oc_0.2"),
    Domain("OPT-QXSLAB", "/workspace/mnt-data/QXSLAB_SAROPT/opt_256_oc_0.2"),
    Domain("HE", "/workspace/mnt-data/HE/train"),
    Domain("IHC", "/workspace/mnt-data/IHC/train"),
    Domain("CT", "/workspace/mnt-data/CT/PNG"),
    Domain("T1", "/workspace/mnt-data/T1-MRI/PNG"),
    Domain("T2", "/workspace/mnt-data/T2-MRI/PNG"),
    Domain("SAR-SEN12", "/workspace/mnt-data/SEN12-splits/train/input"),
    Domain("OPT-SEN12", "/workspace/mnt-data/SEN12-splits/train/target"),
    Domain("ImageNet-1k", "/workspace/mnt-data/imagenet1k_flat"),
]

MODELS_TO_RUN = ["dinov2", "resnet50", "resnet18", "vgg16"]

make_deterministic(SEED)
cache = FeatureCache(OUTPUT_DIR + "/.feature_cache")

In [3]:
def load_features(extractor) -> list[DomainFeatures]:
    loader = DatasetFeatureLoader(
        extractor, pool_size=N_SAMPLES, seed=SEED, cache=cache
    )
    result = []
    for domain in DOMAINS:
        print(f"  Loading: {domain.name}")
        features = loader.load(domain.path, domain.name)
        result.append(DomainFeatures(domain=domain, features=features))
    return result

In [ ]:
def calibrate_epsilon(
    domain_features: list[DomainFeatures],
    cost: str = "sqeuclidean",
    alpha_start: float = 0.1,
    alpha_min: float = 0.001,
    alpha_step: float = 0.5,
) -> dict:
    rng = np.random.default_rng(SEED)

    samples = [
        df.features[
            rng.choice(
                len(df.features), min(N_SAMPLES, len(df.features)), replace=False
            )
        ]
        for df in domain_features
    ]

    # Global mean cost (used as epsilon scale)
    pooled = np.vstack(samples)
    C_global = cdist(pooled, pooled, metric=cost)
    mean_cost = float(np.mean(C_global[C_global > 0]))
    print(f"Calibrating ε for {cost}  (mean_cost={mean_cost:.4f})")

    # Rank all domain pairs by mean inter-domain cost
    n = len(domain_features)
    pair_costs = {
        (i, j): float(np.mean(cdist(samples[i], samples[j], metric=cost)))
        for i in range(n)
        for j in range(i + 1, n)
    }
    sorted_pairs = sorted(pair_costs.items(), key=lambda x: x[1])
    k = min(N_EXTREME, len(sorted_pairs) // 2)
    test_pairs = [p for p, _ in sorted_pairs[:k]] + [p for p, _ in sorted_pairs[-k:]]
    print(
        f"  {len(test_pairs)} extreme pairs  "
        f"(cost range {sorted_pairs[0][1]:.1f} – {sorted_pairs[-1][1]:.1f})"
    )

    # For each extreme pair build a hard calibration subset:
    # pick the N_CALIBRATION rows/cols with the HIGHEST mean cost: worst case for varepsilon
    calibration_pairs = []
    for i, j in test_pairs:
        fi, fj = samples[i], samples[j]
        nc = min(N_CALIBRATION, len(fi), len(fj))
        C = cdist(fi, fj, metric=cost)
        ri = np.argsort(C.mean(axis=1))[-nc:]
        ci = np.argsort(C.mean(axis=0))[-nc:]
        calibration_pairs.append((fi[ri], fj[ci]))

    # Descend alpha until Sinkhorn fails
    best_alpha = best_eps = None
    alpha = alpha_start

    while alpha >= alpha_min:
        eps = alpha * mean_cost
        passed = True

        for da, db in calibration_pairs:
            try:
                m = da.shape[0]
                M = cdist(da, db, metric=cost)
                with np.errstate(over="raise"):
                    w, log = ot.sinkhorn2(
                        ot.unif(m), ot.unif(m), M, eps, method="sinkhorn_log", log=True
                    )
                if np.any(np.isnan(w)) or np.any(np.isinf(w)):
                    passed = False
                    break
                if log.get("err", [0])[-1] > 1e-3:
                    passed = False
                    break
            except Exception:
                passed = False
                break

        print(f"  α={alpha:.4f}  ε={eps:.4f}  {'converged' if passed else 'failed'}")
        if passed:
            best_alpha, best_eps = alpha, eps
            alpha *= alpha_step
        else:
            break

    if best_eps is None:
        raise RuntimeError("Sinkhorn did not converge for any alpha")

    print(f"→ ε={best_eps:.4f}  (α={best_alpha:.4f})")
    return {
        "cost": cost,
        "epsilon": best_eps,
        "alpha": best_alpha,
        "mean_cost": mean_cost,
    }

In [5]:
results = {}

for model_name in MODELS_TO_RUN:
    print(f"\n=== {model_name} ===")
    extractor = None
    try:
        extractor = UniversalFeatureExtractor(model_name)
        results[model_name] = calibrate_epsilon(
            load_features(extractor), cost="sqeuclidean"
        )
    finally:
        del extractor
        torch.cuda.empty_cache()
        gc.collect()


=== dinov2 ===
Initializing Model: DINOV2


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


  Loading: SAR-QXSLAB
  -> cached to /workspace/wasserstein-metric/pretraining_pipeline_results_euclidean_only//.feature_cache/SAR-QXSLAB_dinov2_pool1000_seed42.npy
  Loading: OPT-QXSLAB
  -> cached to /workspace/wasserstein-metric/pretraining_pipeline_results_euclidean_only//.feature_cache/OPT-QXSLAB_dinov2_pool1000_seed42.npy
  Loading: HE
  -> cached to /workspace/wasserstein-metric/pretraining_pipeline_results_euclidean_only//.feature_cache/HE_dinov2_pool1000_seed42.npy
  Loading: IHC
  -> cached to /workspace/wasserstein-metric/pretraining_pipeline_results_euclidean_only//.feature_cache/IHC_dinov2_pool1000_seed42.npy
  Loading: CT
  -> cached to /workspace/wasserstein-metric/pretraining_pipeline_results_euclidean_only//.feature_cache/CT_dinov2_pool1000_seed42.npy
  Loading: T1
  -> cached to /workspace/wasserstein-metric/pretraining_pipeline_results_euclidean_only//.feature_cache/T1_dinov2_pool1000_seed42.npy
  Loading: T2
  -> cached to /workspace/wasserstein-metric/pretraining_p

In [6]:
print("=== Summary ===")
for model_name, info in results.items():
    print(
        f"{model_name:12s}  ε={info['epsilon']:.6f}  α={info['alpha']:.4f}  mean_cost={info['mean_cost']:.4f}"
    )

os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = os.path.join(OUTPUT_DIR, "epsilon_calibration.json")
to_save = [{"backbone": k, **v} for k, v in results.items()]
with open(out_path, "w") as f:
    json.dump(to_save, f, indent=2)
print(f"\nSaved → {out_path}")

=== Summary ===
dinov2        ε=11.267332  α=0.0031  mean_cost=3605.5462
resnet50      ε=2.120524  α=0.0063  mean_cost=339.2838
resnet18      ε=3.591754  α=0.0063  mean_cost=574.6806
vgg16         ε=3.334540  α=0.0250  mean_cost=133.3816

Saved → /workspace/wasserstein-metric/pretraining_pipeline_results_euclidean_only/epsilon_calibration.json
